In [11]:
import pandas as pd
import requests
import os

In [12]:
df=pd.read_csv(r"D:\Intership\data\FT Data - data.csv")
df.head()

,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
3,93626,990175,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
4,286851,526266,hi,522,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...


In [13]:
df_copy=df.head()

### Transcription

In [64]:
import pandas as pd
import requests

all_rows = []
not_work_id = []
working_id = []

def build_url(url, record_id):
    folder_id = url.split("/")[-2]
    return f"https://storage.googleapis.com/upload_goai/{folder_id}/{int(float(record_id))}_transcription.json"

def get_transcription(url, record_id):
    try:
        modified_url = build_url(url, record_id)
        res = requests.get(modified_url, timeout=10)

        if res.status_code == 200:
            working_id.append(record_id)
            return res.json()
        else:
            not_work_id.append(record_id)
            return None

    except:
        not_work_id.append(record_id)
        return None


for url, record_id in zip(df["transcription_url_gcp"], df["recording_id"]):
    
    raw_data = get_transcription(url, record_id)
    if raw_data is not None:
        for seg in raw_data:   
            seg["record_id"] = int(float(record_id))
            all_rows.append(seg)   

                                   
df_final = pd.DataFrame(all_rows)
print("Total rows:", len(df_final))
print(df_final.head())

Total rows: 5941
   start    end  speaker_id  \
0   0.11  14.42      245746   
1  14.42  29.03      245746   
2  29.03  41.84      245746   
3  42.47  50.57      245746   
4  52.70  66.83      245746   

                                                text  record_id  
0  अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...     825780  
1  अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...     825780  
2  जंगल का सफर होता है जब हम रहने के लिए गए थे ना...     825780  
3  पहली बारी था क्योंकि चलना नहीं आता न वहाँ का ज...     825780  
4  हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...     825780  


In [65]:
len(df_final)

5941

In [66]:
df_final.to_csv("collected data/transcription.csv", index=False)

### Audio

In [45]:
data_audio=[]
record_ids=[]
os.makedirs("audio", exist_ok=True)

def build_url(url, record_id):
    folder_id = url.split("/")[-2]
    return f"https://storage.googleapis.com/upload_goai/{folder_id}/{int(float(record_id))}_audio.wav"  
    
def download_audio(audio_url, record_id):
    file_path = f"audio/{record_id}.wav"
    
    # skip if already exists
    if os.path.exists(file_path):
        return file_path

    try:
        audio_url=build_url(url,record_id)
        res = requests.get(audio_url, timeout=20)

        if res.status_code == 200:
            with open(file_path, "wb") as f:
                f.write(res.content)
            return file_path
        else:
            print("Failed:", audio_url)
            return None

    except Exception as e:
        print("Error:", e)
        return None


for url,record_id in zip(df["rec_url_gcp"],df["recording_id"]) :
    raw_data=download_audio(url,record_id)
    if raw_data is not None:
        data_audio.append(raw_data)
        record_ids.append(record_id)

In [ ]:
df_audio = pd.DataFrame(
    data=list(zip(data_audio, record_ids)), 
    columns=["file_path", "record_id"]
)
df_audio.head()


,file_path,record_id
0,audio/825780.wav,825780
1,audio/825727.wav,825727
2,audio/988596.wav,988596
3,audio/990175.wav,990175
4,audio/526266.wav,526266


In [71]:
df_audio.to_csv("collected data/audio_path.csv", index=False)

### Fetch metadata

In [79]:
meta_data=[]
raw_id={}
def build_url(url, record_id):
    folder_id = url.split("/")[-2]
    return f"https://storage.googleapis.com/upload_goai/{folder_id}/{int(float(record_id))}_metadata.json"  

def get_metadata(url,record_id):
    url=build_url(url,record_id)
    try:
        res = requests.get(url, timeout=5)

        if res.status_code == 200:
            return res.json()
        else:
            return None
    except:
        return None

for url,record_id in zip(df["metadata_url_gcp"],df["recording_id"]) :
    raw_data=get_metadata(url,record_id)
    if raw_data is not None:
        raw_data["record_id"]=record_id
        meta_data.append(raw_data)
        

In [82]:

df_meta = pd.DataFrame(meta_data)
print(df_meta.head())

df_meta.to_csv("collected data/metadata.csv", index=False)

   speaker_id gender date_of_birth occupation educational_background  \
0      245746      O    2007-06-23    Student               Graduate   
1      291038      M    2006-04-08    Student              10th Pass   
2      246004      F    2006-12-12    Student              12th Pass   
3       93626      F    2006-08-21    Student              12th Pass   
4      286851      F    2003-06-24    Student               Graduate   

  marital_status native_district native_state current_city  \
0        Married          Cachar        Assam    Nur Sarai   
1      Unmarried       Gopalganj        Bihar    Kalyanpur   
2      Unmarried           Patna        Bihar        Patna   
3      Unmarried       Madhubani        Bihar       Naruar   
4      Unmarried         Cuttack       Odisha      Cuttack   

   years_spent_in_current_district       income device_manufacturer  \
0                                5  0 - 3 Lakhs              Xiaomi   
1                               19  0 - 3 Lakhs     